In [19]:
import os, sys
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import pandas as pd
from tqdm import tqdm

In [20]:
# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

from src.dataset import (
    ButterflyDataset, AugmentedButterflyDataset, get_splits,
    build_label_map, get_train_transform, get_eval_transform, make_dataloader
)
from src.utils import save_checkpoint, load_checkpoint

In [21]:
# ── Device setup ──────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [22]:
# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR       = os.path.abspath("..")
CSV_PATH       = os.path.join(BASE_DIR, "aca-butterflies", "train.csv")
IMG_DIR        = os.path.join(BASE_DIR, "aca-butterflies", "train")
ORACLE_PATH    = os.path.join(BASE_DIR, "saved_models", "oracle_resnet18.pth")

CONF_THRESHOLD = 0.6   # Lowered from 0.8 as ResNet is more "certain" but synth data is imperfect
BATCH_SIZE     = 64
IMAGE_SIZE     = 64
NUM_EPOCHS     = 15    # Enough for fine-tuning ResNet18
LR             = 1e-4

GENERATIVE_MODELS = ["cvae", "gan", "wgan_gp", "biggan", "diffusion"]

In [23]:
def get_oracle_model(num_classes):
    """Returns a pre-trained ResNet18 with modified final layer."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)

In [24]:
def train_oracle():
    """Phase A: Train a strong Oracle on the real training set."""
    label_to_idx, idx_to_label = build_label_map(CSV_PATH)
    num_classes = len(label_to_idx)
    
    if os.path.exists(ORACLE_PATH):
        print(f"Oracle already exists at {ORACLE_PATH}. Loading...")
        model = get_oracle_model(num_classes)
        load_checkpoint(ORACLE_PATH, model, device)
        return model

    print("\n" + "="*50)
    print("PHASE A: Training Oracle (ResNet18) on real data...")
    print("="*50)

    train_df, val_df, _ = get_splits(CSV_PATH, train_ratio=0.8, val_ratio=0.2, seed=42)
    
    train_set = ButterflyDataset(train_df, IMG_DIR, label_to_idx, transform=get_train_transform(IMAGE_SIZE))
    val_set   = ButterflyDataset(val_df,   IMG_DIR, label_to_idx, transform=get_eval_transform(IMAGE_SIZE))
    
    train_loader = make_dataloader(train_set, BATCH_SIZE, shuffle=True)
    val_loader   = make_dataloader(val_set,   BATCH_SIZE, shuffle=False)
    
    model = get_oracle_model(num_classes)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += labels.size(0)
        
        acc = correct / total
        print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}")
        
        if acc > best_acc:
            best_acc = acc
            save_checkpoint(model, ORACLE_PATH, epoch=epoch+1, val_acc=acc)
            
    print(f"Oracle training complete. Best Val Acc: {best_acc:.4f}")
    load_checkpoint(ORACLE_PATH, model, device)
    return model

In [25]:
def filter_images(model):
    """Phase B: Use Oracle to filter generated datasets."""
    print("\n" + "="*50)
    print(f"PHASE B: Filtering generated images (Threshold: {CONF_THRESHOLD})...")
    print("="*50)

    label_to_idx, _ = build_label_map(CSV_PATH)
    model.eval()

    for gen_model in GENERATIVE_MODELS:
        gen_dir = os.path.join(BASE_DIR, "data", "generated", gen_model)
        csv_in  = os.path.join(gen_dir, "generated.csv")
        csv_out = os.path.join(gen_dir, "filtered_generated.csv")

        if not os.path.exists(csv_in):
            print(f"Skipping {gen_model}: {csv_in} not found.")
            continue

        df = pd.read_csv(csv_in)
        dataset = AugmentedButterflyDataset(df, label_to_idx, transform=get_eval_transform(IMAGE_SIZE))
        loader = make_dataloader(dataset, BATCH_SIZE, shuffle=False)

        results = []
        with torch.no_grad():
            for imgs, labels in tqdm(loader, desc=f"Scanning {gen_model}"):
                imgs = imgs.to(device)
                outputs = model(imgs)
                probs = torch.softmax(outputs, dim=1)
                conf, preds = torch.max(probs, dim=1)
                
                for j in range(len(imgs)):
                    predicted_idx = preds[j].item()
                    intended_idx = labels[j].item()
                    confidence = conf[j].item()
                    
                    # Keep if prediction matches intended label AND confidence is high
                    is_kept = (predicted_idx == intended_idx) and (confidence >= CONF_THRESHOLD)
                    results.append(is_kept)

        df_filtered = df[results].copy()
        df_filtered.to_csv(csv_out, index=False)
        
        print(f"  {gen_model.upper()}: Kept {len(df_filtered)}/{len(df)} ({len(df_filtered)/len(df)*100:.1f}%) images.")

In [26]:
if __name__ == "__main__":
    oracle = train_oracle()
    filter_images(oracle)


PHASE A: Training Oracle (ResNet18) on real data...
Epoch 01/15 | Loss: 3.5133 | Val Acc: 0.4916
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 02/15 | Loss: 1.6560 | Val Acc: 0.7192
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 03/15 | Loss: 0.9268 | Val Acc: 0.7744
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 04/15 | Loss: 0.5526 | Val Acc: 0.8089
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 05/15 | Loss: 0.3679 | Val Acc: 0.8276
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 06/15 | Loss: 0.2471 | Val Acc: 0.8335
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_resnet18.pth
Epoch 07/15 | Loss: 0.1669 | Val Acc: 0.8433
[checkpoint] saved → c:\Users\ricar\Downloads\TP2-Students\saved_models\oracle_r

Scanning cvae: 100%|██████████| 30/30 [00:01<00:00, 22.16it/s]


  CVAE: Kept 7/1869 (0.4%) images.


Scanning gan: 100%|██████████| 30/30 [00:01<00:00, 23.06it/s]


  GAN: Kept 94/1869 (5.0%) images.


...

  WGAN_GP: Kept 61/1869 (3.3%) images.


Scanning biggan: 100%|██████████| 30/30 [00:01<00:00, 23.05it/s]


  BIGGAN: Kept 119/1869 (6.4%) images.


Scanning diffusion: 100%|██████████| 30/30 [00:01<00:00, 22.82it/s]

  DIFFUSION: Kept 996/1869 (53.3%) images.
